In [3]:
from src.evaluation import score_answer
import pickle as pkl
import pandas as pd
import os
from pathlib import Path

dataset_path = Path("~/officeqa/")
data_file = "officeqa.csv"
df = pd.read_csv(os.path.join(dataset_path, data_file))
with open('../results/full_run_new_evolved_final.pkl', 'rb') as f:
    data = pkl.load(f)


In [4]:
train_set = {
    'index': [],
    'question': [],
    'predicted': [],
    'ground_truth': [],
    'reasoning': [],
    'score_0.05': [],
    'score_0.01': [],
    'score_0.1': [],
    'score_0.0': [],
    'score_0.025': [],
    'final_score': [],
    'difficulty': [],
    'category': [],
    'source_docs': [],
    'uid': [],
    'source_files': [],
}

In [5]:
for set_index, example in enumerate(data):
    try:
        question = example.question
        index = example.index
        ground_truth = example.ground_truth
        predicted = example.trace.output.final_answer
        reasoning = example.trace.output.reasoning
        final_score = 0.0
        for tolerance in [0.05, 0.01, 0.1, 0.0, 0.025]:
            curr_score = score_answer(ground_truth, predicted, tolerance)
            final_score += curr_score
            train_set['score_'+str(tolerance)].append(curr_score)
        train_set['final_score'].append(final_score/5)
        train_set['index'].append(index)
        train_set['question'].append(question)
        train_set['predicted'].append(predicted)
        train_set['ground_truth'].append(ground_truth)
        train_set['reasoning'].append(reasoning)
        source_docs = df.iloc[index].source_docs
        source_files = df.iloc[index].source_files
        uid = df.iloc[index].uid
        category = df.iloc[index].pseudo_labels
        difficulty = df.iloc[index].difficulty
        train_set['source_docs'].append(source_docs)
        train_set['source_files'].append(source_files)
        train_set['uid'].append(uid)
        train_set['category'].append(category)
        train_set['difficulty'].append(difficulty)
        
        
    except Exception as e:
        print(e)

In [6]:
new_df = pd.DataFrame(train_set)

In [7]:
new_df.head()

,index,question,predicted,ground_truth,reasoning,score_0.05,score_0.01,score_0.1,score_0.0,score_0.025,final_score,difficulty,category,source_docs,uid,source_files
0,211,What was the continously compounded average an...,0.062,-0.063,## Data Extraction\n\nFrom Treasury Bulletin F...,0.0,0.0,0.0,0.0,0.0,0.0,hard,Comparison/Change,https://fraser.stlouisfed.org/title/treasury-b...,UID0212,treasury_bulletin_1964_01.txt
1,133,Using the U.S. Treasury tax and loan account b...,115.61,115.61,I extracted U.S. Treasury tax and loan account...,1.0,1.0,1.0,1.0,1.0,1.0,hard,Statistical Metrics,https://fraser.stlouisfed.org/title/treasury-b...,UID0134,treasury_bulletin_1963_02.txt\r\ntreasury_bull...
2,17,What is the geometric mean of the monthly outl...,81.406,81.406,I extracted monthly US Judiciary outlay data f...,1.0,1.0,1.0,1.0,1.0,1.0,hard,Aggregation,https://fraser.stlouisfed.org/title/treasury-b...,UID0018,treasury_bulletin_1985_03.txt\r\ntreasury_bull...
3,55,"Between the years of 1950 and 1990, in which y...",1975,1973,Based on my comprehensive search through the T...,1.0,1.0,1.0,0.0,1.0,0.8,hard,Comparison/Change,https://fraser.stlouisfed.org/title/treasury-b...,UID0056,treasury_bulletin_1991_09.txt
4,173,Using U.S. Treasury Internal Revenue Collectio...,-3.147,−3.524,From the June 1960 Treasury Bulletin's Interna...,0.0,0.0,0.0,0.0,0.0,0.0,hard,Comparison/Change,https://fraser.stlouisfed.org/title/treasury-b...,UID0174,treasury_bulletin_1960_04.txt\r\ntreasury_bull...


In [8]:
new_df.to_csv("../.dataset/new_runs_evolved/solved_dataset.csv", index=False)

In [10]:
new_df[new_df['final_score'] < 1.0].to_csv("../.dataset/new_runs_evolved/train_set.csv", index=False)